# Football analysis on a Colab GPU

Inference is 87% of this pipeline's runtime (318s of 366s for a 20s clip on
CPU). A GPU removes almost all of that, and more importantly makes
`imgsz=1280` affordable â€” which takes ball detection from about 32% of
frames to 98%.

**Before running anything:** Runtime -> Change runtime type ->
Hardware accelerator -> **T4 GPU** -> Save.

Then work through the cells in order.

## 1. Confirm there is actually a GPU

If this stops with an error, the runtime is still CPU. Everything below
would silently run on CPU and be no faster than your laptop.

In [ ]:
import torch

print('torch', torch.__version__, '| CUDA build:', torch.version.cuda)
print('GPU available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit('No GPU. Runtime -> Change runtime type -> T4 GPU.')
name = torch.cuda.get_device_name(0)
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'device: {name}  ({total:.1f} GB)')

## 2. Mount Google Drive

This is the mounting step. Running it opens an authorisation prompt: pick
your Google account and allow access. Drive then appears at
`/content/drive/MyDrive/` and behaves like any local folder.

The mount lasts for this session only â€” re-run it after a disconnect.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 3. Settings

Code comes from GitHub. Only the two things git cannot hold live in Drive:

```
MyDrive/football_analysis/
    best.pt                     <- model weights, 165 MB
    videos/city_vs_spurs.mp4    <- whichever clip you want
```

Upload those once and they stay put; the code refreshes on every clone.

In [ ]:
import os

REPO = 'https://github.com/juddooooooooo/football_analysis_computer_vision.git'
DRIVE = '/content/drive/MyDrive/football_analysis'
VIDEO_NAME = 'city_vs_spurs.mp4'
START, END = '14', '34'
IMGSZ = 1280      # 640 matches the local runs; 1280 is what finds the ball
BATCH = 32        # raise if the GPU has memory spare

WORK = '/content/football'

for path in (DRIVE, f'{DRIVE}/best.pt', f'{DRIVE}/videos/{VIDEO_NAME}'):
    print(('found  ' if os.path.exists(path) else 'MISSING'), path)

### Memory budget

Frames are held in RAM and the annotated copy doubles that. This is the
most common way a run dies on Colab, so check before a long section.

In [ ]:
import cv2, os

cap = cv2.VideoCapture(DRIVE + '/videos/' + VIDEO_NAME)
fps = cap.get(cv2.CAP_PROP_FPS)
w, h = int(cap.get(3)), int(cap.get(4))
cap.release()

n = (float(END) - float(START)) * fps
raw = n * w * h * 3 / 1e9
total = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1e9
print('%s: %dx%d @ %.0f fps' % (VIDEO_NAME, w, h, fps))
print('section %s-%ss = %.0f frames = %.1f GB raw, about %.1f GB peak'
      % (START, END, n, raw, raw * 2))
print('this machine has %.1f GB RAM' % total)
if raw * 2 > total * 0.65:
    safe = (total * 0.65 / 2) / (w * h * 3 / 1e9) / fps
    print('TOO BIG. Use about %.0fs: set END = %.0f' % (safe, float(START) + safe))
else:
    print('fits comfortably')


## 4. Dependencies

Colab already ships a CUDA build of torch, so nothing here may reinstall
it â€” that is the usual way a GPU session quietly ends up on CPU. The
assertion afterwards confirms torch survived.

In [ ]:
!pip -q install ultralytics supervision

import torch
print('torch still CUDA-enabled:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'A dependency replaced torch with a CPU build.'

## 5. Clone the code, link the assets

Weights and video are symlinked rather than copied, so nothing large moves.

If the repo is private the clone fails here â€” either make it public, or
clone using a personal access token.

In [ ]:
import os, shutil

shutil.rmtree(WORK, ignore_errors=True)
!git clone -q {REPO} {WORK}

for sub in ('models', 'input_videos', 'output_videos', 'stubs'):
    os.makedirs(f'{WORK}/{sub}', exist_ok=True)

def link(src, dst):
    if os.path.exists(src) and not os.path.exists(dst):
        os.symlink(src, dst)

link(f'{DRIVE}/best.pt', f'{WORK}/models/best.pt')
link(f'{DRIVE}/videos/{VIDEO_NAME}', f'{WORK}/input_videos/{VIDEO_NAME}')

os.chdir(WORK)
print('working in', os.getcwd())
print('weights linked:', os.path.exists('models/best.pt'))
print('video linked  :', os.path.exists(f'input_videos/{VIDEO_NAME}'))

## 6. Benchmark before committing to a long run

Measures real throughput on this GPU at both resolutions and counts how
often the ball is found. Local CPU baseline for comparison:
**0.53 s/frame at 640, 2.84 s/frame at 1280.**

In [ ]:
import time
import cv2
import torch
from ultralytics import YOLO

cap = cv2.VideoCapture(f'input_videos/{VIDEO_NAME}')
fps = cap.get(cv2.CAP_PROP_FPS)
cap.set(cv2.CAP_PROP_POS_FRAMES, int(float(START) * fps))
frames = []
for _ in range(60):
    ok, f = cap.read()
    if not ok:
        break
    frames.append(f)
cap.release()
print(f'{len(frames)} frames at {frames[0].shape[1]}x{frames[0].shape[0]}')

model = YOLO('models/best.pt')
ball_id = [k for k, v in model.names.items() if v == 'ball'][0]

for size in (640, 1280):
    model.predict(frames[:8], imgsz=size, device=0, half=True, verbose=False)
    torch.cuda.synchronize()
    t0 = time.time()
    out = model.predict(frames, imgsz=size, conf=0.1, device=0, half=True,
                        batch=BATCH, verbose=False)
    torch.cuda.synchronize()
    per = (time.time() - t0) / len(frames)
    hits = sum(1 for r in out if (r.boxes.cls == ball_id).sum() > 0)
    cpu = 0.53 if size == 640 else 2.84
    print(f'imgsz={size}: {per*1000:6.1f} ms/frame  ({cpu/per:5.1f}x the CPU)'
          f'   ball in {hits}/{len(frames)} = {100*hits/len(frames):.0f}%')

## 7. Run the pipeline

Detections are cached in `stubs/` keyed by video, section **and** imgsz, so
re-running only the drawing stages is quick.

The section needs a calibration. `calibrations.json` is in the repo, so
anything calibrated locally works straight away â€” currently
`city_vs_spurs 14-34`, `brazil_mexico 15-35` and
`yt_download.f398 1384-1415`. A new section has to be calibrated first.

In [ ]:
import subprocess

cmd = ['python', 'main.py', 'input_videos/' + VIDEO_NAME,
       '--start', START, '--end', END,
       '--device', '0', '--half',
       '--imgsz', str(IMGSZ), '--batch', str(BATCH)]
print(' '.join(cmd), flush=True)

proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout[-4000:])
if proc.returncode != 0:
    print('--- STDERR ---')
    print(proc.stderr[-4000:])
    print('--- exited %d ---' % proc.returncode)
    if proc.returncode < 0:
        print('Killed by a signal, which on Colab is almost always RAM.')
        print('Shorten the section in cell 3 and re-run from there.')
else:
    print('--- finished cleanly ---')


## 8. Watch the result here

The pipeline writes AVI, which browsers will not play, so this converts a
copy to mp4 for inline viewing. Colab already has ffmpeg.

In [ ]:
import glob, os
from base64 import b64encode
from IPython.display import HTML

avis = sorted(glob.glob('output_videos/*.avi'), key=os.path.getmtime)
if not avis:
    print('No output video, so the run in the previous cell did not finish.')
    print('output_videos contains:', os.listdir('output_videos') or '(empty)')
    print()
    print('Most likely causes, in order:')
    print(' 1. Out of RAM. Colab free tier gives about 12.7 GB; a 20s clip at')
    print('    1920x1080 peaks near 10 GB once the annotated copy exists.')
    print('    Fix: shorten the section in cell 3, then re-run from there.')
    print(' 2. The clone or symlinks failed. Re-check the cell 5 output.')
    print(' 3. Scroll up: the run cell prints the real error now.')
else:
    avi = avis[-1]
    mp4 = avi.replace('.avi', '_preview.mp4')
    os.system('ffmpeg -y -loglevel error -i "%s" -vcodec libx264 -crf 28 "%s"' % (avi, mp4))
    print('%s  ->  %s  (%.1f MB)' % (avi, mp4, os.path.getsize(mp4)/1e6))
    data = b64encode(open(mp4, 'rb').read()).decode()
    display(HTML('<video width=900 controls><source src="data:video/mp4;base64,%s" type="video/mp4"></video>' % data))


## 9. Copy results back to Drive

Colab storage is wiped when the session ends. The stubs are worth keeping â€”
they are the expensive part.

In [ ]:
import glob, os, shutil

for sub in ('output_videos', 'stubs'):
    dest = f'{DRIVE}/{sub}'
    os.makedirs(dest, exist_ok=True)
    for src in glob.glob(f'{sub}/*'):
        if os.path.isfile(src):
            shutil.copy2(src, dest)
            print('copied', os.path.basename(src))

print('\nsaved to', DRIVE)